# Results Notebook

Code here replicates graphs presented in the report. Follow instructions under load data
to determine which experiment to replicate and note any further instructions through
notebook.

In [ ]:

import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from mizani.formatters import percent_format
from plotnine import (
    aes,
    element_text,
    facet_grid,
    facet_wrap,
    geom_boxplot,
    geom_label,
    geom_point,
    geom_smooth,
    geom_tile,
    ggplot,
    labs,
    scale_y_continuous,
    theme,
    theme_bw,
)

from dataset_similarity.constants import (
    DATA_CONFIG_DIR,
    EVAL_RESULT_DIR,
    EXPERIMENT_CONFIG_DIR,
    METRICS_RESULT_DIR,
    PROJECT_DIR,
)
from dataset_similarity.utils import load_yaml_from_path

In [ ]:

UNLABELLED_METRIC_LABEL_MAP = {
    "mmd": "MMD",
    "ot_exact": "OT (Exact)",
    "ot_sinkhorn": "OT (Sinkhorn)",
}

LABELLED_METRIC_LABEL_MAP = {
    "otdd_approx": "OTDD (Approx)",
    "otdd_exact": "OTDD (Exact)",
    "otce_ot_sinkhorn_both": "OTCE OT (Sinkhorn)",
    "otce_ot_sinkhorn_coupling": "F-OTCE OT (Sinkhorn)",
    "otce_otdd_both": "OTCE OTDD",
    "otce_otdd_coupling": "F-OTCE OTDD",
}

# fn for extending the DV_LABEL_MAP to include macro and micro versions of the metrics
def _insert_middle(string: str, word: str, separator: str = "_") -> str:
    words = string.split(separator)
    start = words[0:len(words)-1]
    if not isinstance(start, list):
        start = [start]
    end = [words[len(words)-1]]
    return separator.join([*start, word, *end])

DV_LABEL_MAP = {
    "accuracy_difference": "ΔAccuracy",
    "average_precision_difference": "ΔAP",
    "precision_difference": "ΔPrecision",
    "recall_difference": "ΔRecall",
    "f1_difference": "ΔF1",
    "roc_auc_difference": "ΔROC AUC",
}
DV_LABEL_MAP = {
    k: v
    for metric, label in DV_LABEL_MAP.items()
    for k, v in {
        metric: label,
        _insert_middle(metric, "macro"): label + " (Macro)",
        _insert_middle(metric, "micro"): label + " (Micro)",
    }.items()
}

## Load data

First, fill the list here with the experiment names you would like to load and produce
results for.

Comment out or add in a new set of experiment names to determine which experiment(s) to
produce plots for.

In [ ]:
# experiment_names = ["experiment_1_main"]
# experiment_names = ["experiment_2_balance"]
experiment_names = ["experiment_3_multilabel"]
# experiment_names = ["experiment_4_ood_positive"]
# experiment_names = ["experiment_5_resnet"]

# The following determines the output name
output_name = experiment_names[0]
if len(experiment_names) > 1:
    output_name = experiment_names[0] + "_with_extras"

Next, run the following cells

In [ ]:
def load_json(result_path):
    """Wrapper around json.load"""
    with open(result_path) as f:
        return json.load(f)


def get_task_id(task_name: str) -> int:
    """Get the task id from the task name"""
    return int(task_name.split("_")[-1])

digit_re = re.compile(r"metrics_\d+\.yaml$")

def load_experiment_results(experiment_name, store_size=None) -> pd.DataFrame:
    eval_results = pd.DataFrame(
        load_json(path)
        for path in (EVAL_RESULT_DIR / experiment_name).glob("finetune_*_results.json")
    )
    metric_paths = [
        p for p in (METRICS_RESULT_DIR / experiment_name).glob("metrics_*.yaml")
        if digit_re.match(p.name)
    ]
    if store_size is not None:
        metric_paths = list(
            (
                METRICS_RESULT_DIR
                / experiment_name
            ).glob(f"metrics_*_asymptotics/store_size_{store_size}.yaml")
        )
    metrics_results = pd.DataFrame(load_yaml_from_path(path) for path in metric_paths)
    if store_size is not None:
        metrics_results["store_size"] = store_size
    eval_results["id"] = eval_results.name.apply(get_task_id)
    metrics_results["id"] = metrics_results.dataset1.apply(get_task_id)
    results = pd.merge(eval_results, metrics_results, on="id", how="inner")
    return results.sort_values(by="id").reset_index(drop=True)


def load_metrics_used(experiment_names: list[str]) -> list[str]:
    metrics = []
    for experiment_name in experiment_names:
        cfg = load_yaml_from_path(EXPERIMENT_CONFIG_DIR / f"{experiment_name}.yaml")
        metrics += cfg["metrics"]
    return list(set(metrics))


def load_all_results(
    experiment_names: list[str], store_size: int | None = None
) -> tuple[pd.DataFrame, list[str]]:
    return pd.concat(
        [load_experiment_results(name, store_size=store_size) for name in experiment_names],  # noqa: E501
        ignore_index=True,
    ), load_metrics_used(experiment_names)


def melt_to_plot_df(results_df: pd.DataFrame, metrics: list[str]) -> pd.DataFrame:
    eval_metrics = results_df.columns[results_df.columns.str.endswith("_difference")]
    id_vars_1 = ["id", "name", *eval_metrics]
    id_vars_2 = ["id", "name", "metric", "value"]
    if "store_size" in results_df.columns:
        id_vars_1.append("store_size")
        id_vars_2.append("store_size")
    return results_df.melt(
        id_vars=id_vars_1,
        value_vars=metrics,
        var_name="metric",
        value_name="value"
    ).melt(
        id_vars=id_vars_2,
        value_vars=eval_metrics,
        var_name="dv",
        value_name="dv_value"
    ).assign(
        metric=lambda x: x.metric.map(
            {**UNLABELLED_METRIC_LABEL_MAP, **LABELLED_METRIC_LABEL_MAP}
        ),
    ).assign(
        dv=lambda x: x.dv.map(DV_LABEL_MAP),
    )


def load_task_attributes(task_path: Path) -> dict:
    cfg = load_yaml_from_path(task_path)
    kwargs = cfg["kwargs"]
    kwargs["name"] = (
        task_path.parent.stem + "/" + task_path.stem.replace("testARC", "finetune")
    )
    return {
        k: v[0] if isinstance(v, list) and len(v) == 1 else v for k, v in kwargs.items()
    }


def load_experiment_task_attributes(experiment_name: str) -> list[dict]:
    task_paths = (DATA_CONFIG_DIR / experiment_name).glob("testARC_*.yaml")
    return pd.DataFrame(load_task_attributes(path) for path in task_paths)

In [ ]:
results, metrics = load_all_results(experiment_names)
task_data = pd.concat(
    load_experiment_task_attributes(name) for name in experiment_names
)

In [ ]:

pdf = melt_to_plot_df(results, metrics)
pdf = pd.merge(pdf, task_data, on="name", how="left")

In [ ]:
if output_name == "experiment_3_multilabel":
    underscored = pdf.positive_class.apply("_".join)
    underscored_keys = {key: str(i) for i, key in enumerate(underscored.unique())}
    pdf["positive_class_group"] = underscored.apply(lambda x: underscored_keys[x])

DIAGNOSTICS

In [ ]:
pdf[pdf.dv_value.isna()].name.unique()

In [ ]:
pdf[pdf.value.isna()].name.unique()

## Main Result Plots

In [ ]:
def plot_metrics_vs_ap_difference_colour(
    pdf: pd.DataFrame,
    group: str,
    group_label: str | None = None,
    use_unlabelled=False,
    use_labelled=False,
    eval_metric: str | None = None
):
    metric_labels = []
    if use_unlabelled:
        metric_labels += list(UNLABELLED_METRIC_LABEL_MAP.values())
    if use_labelled:
        metric_labels += list(LABELLED_METRIC_LABEL_MAP.values())
    if len(metric_labels) == 0:
        msg = "At least one of use_unlabelled or use_labelled must be True"
        raise ValueError(msg)
    pdf = pdf[pdf.metric.isin(metric_labels)]
    N = len(pdf.metric.unique())
    group_label = group_label or group
    if eval_metric is None:
        eval_metric = (
            "ΔAP"
            if "ΔAP" in pdf.dv.unique()
            else "ΔAP (Macro)"
        )
    return (
        ggplot(
            pdf[pdf.dv == eval_metric],
            aes(x="value", y="dv_value"),
        )
        + geom_point(aes(color=group, shape=group))
        + geom_smooth(method="lm", color="black", se=True)
        + facet_wrap("~metric", scales="free", ncol=N)
        + labs(
            x="Metric Value",
            y=eval_metric,
            color=group_label,
            shape=group_label,
        )
        + theme_bw()
        + theme(
            axis_text_x=element_text(rotation=45, hjust=1),
            figure_size=(N*2 + 2, 3),
            legend_position="right",
            subplots_adjust={"right": 2 / (N*3 + 2)},
        )
    )

In [ ]:
if output_name not in ["experiment_3_multilabel", "experiment_4_ood_positive"]:
    p = plot_metrics_vs_ap_difference_colour(
        pdf, group="positive_class", group_label="Positive Class", use_unlabelled=True
    )
elif output_name == "experiment_3_multilabel":
    p = plot_metrics_vs_ap_difference_colour(
        pdf, group="positive_class_group", group_label="Positive Class Group", use_unlabelled=True  # noqa: E501
    )
p.save(
    PROJECT_DIR / f"plots/{output_name}_metrics_vs_ap_difference_unlabelled_positive_class.pdf",  # noqa: E501
    dpi=300,
)
p

In [ ]:
if output_name not in ["experiment_3_multilabel", "experiment_4_ood_positive"]:
    p = plot_metrics_vs_ap_difference_colour(
        pdf, group="positive_class", group_label="Positive Class", use_labelled=True
    )
    p.save(
        PROJECT_DIR / f"plots/{output_name}_metrics_vs_ap_difference_labelled_positive_class.pdf",  # noqa: E501
        dpi=300,
    )
    display(p)

## Grid of Metrics

In [ ]:
def plot_metrics_vs_dv(pdf: pd.DataFrame, use_unlabelled=False, use_labelled=False):
    metric_labels = []
    if use_unlabelled:
        metric_labels += list(UNLABELLED_METRIC_LABEL_MAP.values())
    if use_labelled:
        metric_labels += list(LABELLED_METRIC_LABEL_MAP.values())
    if len(metric_labels) == 0:
        msg = "At least one of use_unlabelled or use_labelled must be True"
        raise ValueError(msg)
    pdf = pdf[pdf.metric.isin(metric_labels)]
    N = len(pdf.metric.unique())
    M = len(pdf.dv.unique())
    return (
        ggplot(
            pdf,
            aes(x="value", y="dv_value"),
        )
        + geom_point()
        + geom_smooth(method="lm", color="red")
        + facet_grid("dv~metric", scales="free")
        + labs(
            x="Metric Value",
            y="Eval Metric Value",
        )
        + theme_bw()
        + theme(
            axis_text_x=element_text(rotation=45, hjust=1),
            figure_size=(N*2, M*1.8),
            # aspect_ratio=1.0,
        )
    )

In [ ]:
p = plot_metrics_vs_dv(pdf, use_unlabelled=True)
p.save(
    PROJECT_DIR / f"plots/{output_name}_metrics_vs_outcome_grid_unlabelled_positive_class.pdf",  # noqa: E501
    dpi=300,
)
p

In [ ]:
if output_name != "experiment_3_multilabel":
    p = plot_metrics_vs_dv(pdf, use_labelled=True)
    p.save(
        PROJECT_DIR / f"plots/{output_name}_metrics_vs_outcome_grid_labelled_positive_class.pdf",  # noqa: E501
        dpi=300,
    )
    display(p)

## Correlations

In [ ]:
corr_vars = {
    "average_precision_difference": "ΔAP",
    "average_precision_macro_difference": "ΔAP (Macro)",
    **UNLABELLED_METRIC_LABEL_MAP,
    **LABELLED_METRIC_LABEL_MAP,
}

def build_corr_plot(df, corr="pearson"):
    vars = {k: v for k, v in corr_vars.items() if k in df.columns}
    corr_pdf = (
        df[vars.keys()]
        .rename(columns=vars)
        .corr(corr)
        .melt(ignore_index=False)
        .reset_index()
        .set_axis(["metric_1", "metric_2", "correlation"], axis=1)
        .assign(lab_text=lambda x: x.correlation.map(lambda v: f"{v:.2f}"))
    )
    corr_pdf.metric_1 = pd.Categorical(
        corr_pdf.metric_1,
        categories=vars.values(),
        ordered=True,
    )
    corr_pdf.metric_2 = pd.Categorical(
        corr_pdf.metric_2,
        categories=reversed(vars.values()),
        ordered=True,
    )
    title = (
        "Pearson's Correlation Matrix" if corr == "pearson"
        else "Spearman's Correlation Matrix"
    )
    return (
        ggplot(
            corr_pdf,
            aes(
                x="metric_1",
                y="metric_2",
                fill="correlation",
                label="lab_text",
            )
        )
        + geom_tile()
        + geom_label(fill="white", size=8)
        + labs(title=title, x="", y="")
        + theme_bw()
        + theme(
            axis_text_x = element_text(rotation=45, hjust=1),
            aspect_ratio=1,
        )
    )

In [ ]:
p = build_corr_plot(results, "pearson")
p.save(PROJECT_DIR / f"plots/{output_name}_pearson_corr.pdf", dpi=300)
p

In [ ]:
p = build_corr_plot(results, "spearman")
p.save(PROJECT_DIR / f"plots/{output_name}_spearmans_corr.pdf", dpi=300)
p

## Asymptotics

In [ ]:
if output_name == "experiment_1_main":
    asymptotic_results = pd.concat(
        load_all_results(experiment_names, store_size=size)[0]
        for size in [500, 1000, 2500, 3000, 5000, 6000, 12000, 20000, 30000, None]
    ).reset_index(drop=True)
    asymptotic_pdf = melt_to_plot_df(asymptotic_results, metrics)
    asymptotic_pdf = pd.merge(asymptotic_pdf, task_data, on="name", how="left")
    asymptotic_pdf.store_size = asymptotic_pdf.store_size.fillna(60000)
    # NB plots in report don't use 60k value - if this changes, it needs fixing to
    # correct value

### Pure Metric Convergence

In [ ]:
if output_name == "experiment_1_main":
    asymptotics = asymptotic_pdf[asymptotic_pdf.store_size != 60000]
    asymptotics_pdf2 = pd.merge(
        asymptotics,
        pdf[["id", "metric", "value"]].rename(columns={"value": "full_value"}),
        on=["id", "metric"],
    )
    asymptotics_pdf2 = asymptotics_pdf2[asymptotics_pdf2.dv == "ΔAP"]
    asymptotics_pdf2["metric_difference"] = asymptotics_pdf2["value"] - asymptotics_pdf2["full_value"]  # noqa: E501
    asymptotics_pdf2["metric_difference_ratio"] = asymptotics_pdf2["metric_difference"] / asymptotics_pdf2["full_value"]  # noqa: E501

    asymptotics_pdf2["store_size"] = pd.Categorical(
        asymptotics_pdf2["store_size"],
        categories=[500, 1000, 2500, 3000, 5000, 6000, 12000, 20000, 30000, 60000],
        ordered=True
    )

In [ ]:
def plot_asymptotic_boxplots(
    pdf, difference, difference_label=None, scale_y=None,
):
    metric_label = difference_label or difference
    scales = "free_y" if scale_y is None else "fixed"
    p = (
        ggplot(
            pdf[pdf.metric.isin(list(UNLABELLED_METRIC_LABEL_MAP.values()))],
            aes(y=difference, x="store_size")
        )
        + geom_boxplot(notch=True)
        + facet_wrap("~metric", ncol=3, scales=scales)
        + theme_bw()
        + theme(
            axis_text_x = element_text(rotation=45, hjust=1),
            figure_size=(6, 3),
        )
        + labs(
            x="Store Size",
            y=metric_label,
        )
    )
    if scale_y is not None:
        p += scale_y_continuous(labels=percent_format(), limits=scale_y)
    return p

In [ ]:
if output_name == "experiment_1_main":
    p = plot_asymptotic_boxplots(
        asymptotics_pdf2,
        difference="metric_difference",
        difference_label="Metric Difference",
    )
    p.save(
        PROJECT_DIR / f"plots/{output_name}_asymptotics_absolute.pdf",
        dpi=300,
    )
    display(p)

In [ ]:
if output_name == "experiment_1_main":
    p = plot_asymptotic_boxplots(
        asymptotics_pdf2,
        difference="metric_difference_ratio",
        difference_label="Relative Measure Change",
        scale_y=[-.1,.3]
    )
    p.save(
        PROJECT_DIR / f"plots/{output_name}_asymptotics_relative.pdf",
        dpi=300,
    )
    display(p)

### Repeat Thresholds Plot

In [ ]:
def plot_asymptotic_thresholds(
    pdf: pd.DataFrame,
    group: str,
    group_label: str | None = None,
    use_unlabelled=False,
    use_labelled=False,
):
    metric_labels = []
    if use_unlabelled:
        metric_labels += list(UNLABELLED_METRIC_LABEL_MAP.values())
    if use_labelled:
        metric_labels += list(LABELLED_METRIC_LABEL_MAP.values())
    if len(metric_labels) == 0:
        msg = "At least one of use_unlabelled or use_labelled must be True"
        raise ValueError(msg)
    pdf = pdf[pdf.metric.isin(metric_labels)]
    N = len(pdf.metric.unique())
    M = len(pdf.store_size.unique())
    group_label = group_label or group
    ap_name = (
        "ΔAP"
        if "ΔAP" in pdf.dv.unique()
        else "ΔAP (Macro)"
    )
    return (
        ggplot(
            pdf[pdf.dv == ap_name],
            aes(x="value", y="dv_value"),
        )
        + geom_point(aes(color=group, shape=group))
        + geom_smooth(method="lm", color="black", se=True)
        + facet_grid("store_size~metric", scales="free")
        + labs(
            x="Metric Value",
            y=ap_name,
            color=group_label,
            shape=group_label,
        )
        + theme_bw()
        + theme(
            axis_text_x=element_text(rotation=45, hjust=1),
            figure_size=(N*2 + 2, M*2),
            legend_position="right",
            subplots_adjust={"right": 2 / (N*3 + 2)},
        )
    )

In [ ]:
if output_name == "experiment_1_main":
    p = plot_asymptotic_thresholds(
        asymptotic_pdf, group="positive_class", group_label="Positive Class", use_unlabelled=True  # noqa: E501
    )
    p.save(
        PROJECT_DIR / f"plots/{output_name}_asymptotic_thresholds.pdf",
        dpi=300,
    )
    display(p)

## Grouping Plots

In [ ]:
pdf["negative_superclass_label"] = pd.Categorical(
    (
        pdf
        .negative_superclass
        .apply(lambda x: x if x is not None else "None")
        .map({"None": "None", "vehicle": "Vehicle", "food": "Food"})
    ),
    categories=["None", "Vehicle", "Food"],
)

pdf["positive_fraction_label"] = pd.Categorical(
    (
        pdf
        .positive_fraction
        .apply(lambda x: x if not np.isnan(x) else "Default")
        .map({
            "Default": "Default",
            .05: "0.05",
            .1: "0.1",
            .2: "0.2",
            .3: "0.3",
            .4: "0.4",
            .5: "0.5",
        })
    ),
    categories=["Default", "0.05", "0.1", "0.2", "0.3", "0.4", "0.5"],
)

if output_name == "experiment_1_main":
    pdf["max_objects_label"] = pd.Categorical(
        (
            pdf
            .max_objects_per_image
            .apply(lambda x: x if not np.isnan(x) else "None")
            .map({"None": "Inf", 4.: "4"})
        ),
        categories=["Inf", "4"],
    )

    pdf["filter_class_label"] = pd.Categorical(
        (
            pdf
            .filter_class
            .apply(lambda x: x if x is not None else "None")
            .map({"None": "All", "positive": "Positive"})
        ),
        categories=["All", "Positive"],
    )


In [ ]:
if output_name not in ["experiment_3_multilabel", "experiment_4_ood_positive"]:
    p = plot_metrics_vs_ap_difference_colour(
        pdf,
        group="negative_superclass_label",
        group_label="Negative Superclass",
        use_unlabelled=True,
    )
    p.save(
        PROJECT_DIR / f"plots/{output_name}_metrics_vs_ap_diff_negative_super.pdf",
        dpi=300,
    )
    display(p)

    p = plot_metrics_vs_ap_difference_colour(
        pdf,
        group="positive_fraction_label",
        group_label="Positive Cls Fraction",
        use_unlabelled=True,
    )
    p.save(
        PROJECT_DIR / f"plots/{output_name}_metrics_vs_ap_diff_positive_fraction.pdf",
        dpi=300,
    )
    display(p)


if output_name == "experiment_1_main":
    p = plot_metrics_vs_ap_difference_colour(
        pdf,
        group="max_objects_label",
        group_label="Max Objects per Image",
        use_unlabelled=True,
    )
    p.save(
        PROJECT_DIR / f"plots/{output_name}_metrics_vs_ap_diff_max_objects.pdf",
        dpi=300,
    )
    display(p)

    p = plot_metrics_vs_ap_difference_colour(
        pdf,
        group="filter_class_label",
        group_label="Filtered Images",
        use_unlabelled=True,
    )
    p.save(
        PROJECT_DIR / f"plots/{output_name}_metrics_vs_ap_diff_filtered_images.pdf",
        dpi=300,
    )
    display(p)